In [2]:
# Import all required libraries
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import numpy as np
import matplotlib.pyplot as plt
import os
print("TensorFlow Version:", tf.__version__)
print("All libraries imported successfully!")

TensorFlow Version: 2.21.0
All libraries imported successfully!


In [3]:
# Dataset paths
TRAIN_DIR = r"C:/plant_disease_project/dataset_final/train"
VALID_DIR = r"C:/plant_disease_project/dataset_final/valid"

# Parameters
IMG_SIZE = 224        # Image size for MobileNetV2
BATCH_SIZE = 32       # Number of images per batch
EPOCHS = 10           # Number of training rounds
NUM_CLASSES = 14      # Our 14 disease classes

print("Train path:", TRAIN_DIR)
print("Valid path:", VALID_DIR)
print("Number of classes:", NUM_CLASSES)

Train path: C:/plant_disease_project/dataset_final/train
Valid path: C:/plant_disease_project/dataset_final/valid
Number of classes: 14


In [4]:
# Image Data Generators
train_datagen = ImageDataGenerator(
    rescale=1./255,           # Normalize pixel values
    rotation_range=20,        # Random rotation
    horizontal_flip=True,     # Random flip
    zoom_range=0.2,           # Random zoom
    shear_range=0.2,          # Random shear
    fill_mode='nearest'       # Fill empty pixels
)

valid_datagen = ImageDataGenerator(
    rescale=1./255            # Only normalize, no augmentation
)

# Load training data
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# Load validation data
valid_generator = valid_datagen.flow_from_directory(
    VALID_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

print("Class Names:", list(train_generator.class_indices.keys()))

Found 26726 images belonging to 14 classes.
Found 6682 images belonging to 14 classes.
Class Names: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___healthy']


In [4]:
# Load MobileNetV2 base model (pre-trained on ImageNet)
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze base model layers
base_model.trainable = False

# Build complete model
model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
])

# Compile model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()
print("Model built successfully!")

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 7s 1us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224 (Functional)    │ (None, 7, 7, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         163,968 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 14)                  │           1,806 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,423,758 (9.25 MB)

 Trainable params: 165,774 (647.55 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Model built successfully!


In [5]:
# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    'model/plant_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

# Train the model
print("Training started... Please wait!")
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=valid_generator,
    callbacks=[early_stop, checkpoint]
)

print("Training Complete!")

Training started... Please wait!
Epoch 1/10
836/836 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7513 - loss: 0.7617
Epoch 1: val_accuracy improved from None to 0.95136, saving model to model/plant_model.h5



Epoch 1: finished saving model to model/plant_model.h5
836/836 ━━━━━━━━━━━━━━━━━━━━ 1597s 2s/step - accuracy: 0.8545 - loss: 0.4364 - val_accuracy: 0.9514 - val_loss: 0.1555
Epoch 2/10
836/836 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9265 - loss: 0.2099
Epoch 2: val_accuracy improved from 0.95136 to 0.95720, saving model to model/plant_model.h5



Epoch 2: finished saving model to model/plant_model.h5
836/836 ━━━━━━━━━━━━━━━━━━━━ 942s 1s/step - accuracy: 0.9306 - loss: 0.2001 - val_accuracy: 0.9572 - val_loss: 0.1233
Epoch 3/10
836/836 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9415 - loss: 0.1644
Epoch 3: val_accuracy did not improve from 0.95720
836/836 ━━━━━━━━━━━━━━━━━━━━ 935s 1s/step - accuracy: 0.9421 - loss: 0.1625 - val_accuracy: 0.9559 - val_loss: 0.1287
Epoch 4/10
836/836 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9470 - loss: 0.1512
Epoch 4: val_accuracy did not improve from 0.95720
836/836 ━━━━━━━━━━━━━━━━━━━━ 950s 1s/step - accuracy: 0.9467 - loss: 0.1481 - val_accuracy: 0.9560 - val_loss: 0.1231
Epoch 5/10
836/836 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9497 - loss: 0.1388
Epoch 5: val_accuracy improved from 0.95720 to 0.96827, saving model to model/plant_model.h5



Epoch 5: finished saving model to model/plant_model.h5
836/836 ━━━━━━━━━━━━━━━━━━━━ 941s 1s/step - accuracy: 0.9520 - loss: 0.1370 - val_accuracy: 0.9683 - val_loss: 0.0969
Epoch 6/10
836/836 ━━━━━━━━━━━━━━━━━━━━ 0s 705ms/step - accuracy: 0.9571 - loss: 0.1217
Epoch 6: val_accuracy did not improve from 0.96827
836/836 ━━━━━━━━━━━━━━━━━━━━ 659s 788ms/step - accuracy: 0.9569 - loss: 0.1231 - val_accuracy: 0.9680 - val_loss: 0.0911
Epoch 7/10
836/836 ━━━━━━━━━━━━━━━━━━━━ 0s 685ms/step - accuracy: 0.9606 - loss: 0.1127
Epoch 7: val_accuracy improved from 0.96827 to 0.96917, saving model to model/plant_model.h5



Epoch 7: finished saving model to model/plant_model.h5
836/836 ━━━━━━━━━━━━━━━━━━━━ 636s 760ms/step - accuracy: 0.9573 - loss: 0.1227 - val_accuracy: 0.9692 - val_loss: 0.0950
Epoch 8/10
836/836 ━━━━━━━━━━━━━━━━━━━━ 0s 698ms/step - accuracy: 0.9598 - loss: 0.1154
Epoch 8: val_accuracy did not improve from 0.96917
836/836 ━━━━━━━━━━━━━━━━━━━━ 645s 772ms/step - accuracy: 0.9591 - loss: 0.1162 - val_accuracy: 0.9690 - val_loss: 0.0919
Epoch 9/10
836/836 ━━━━━━━━━━━━━━━━━━━━ 0s 767ms/step - accuracy: 0.9626 - loss: 0.1070
Epoch 9: val_accuracy improved from 0.96917 to 0.97097, saving model to model/plant_model.h5



Epoch 9: finished saving model to model/plant_model.h5
836/836 ━━━━━━━━━━━━━━━━━━━━ 706s 845ms/step - accuracy: 0.9625 - loss: 0.1058 - val_accuracy: 0.9710 - val_loss: 0.0913
Training Complete!


In [5]:
from tensorflow.keras.models import load_model

# Load the trained model
model = load_model('model/plant_model.h5')

# Get class names (same order as training)
class_names = list(train_generator.class_indices.keys())

print("Model loaded successfully!")
print("Classes:", class_names)

Model loaded successfully!
Classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___healthy']


In [6]:
from tensorflow.keras.preprocessing import image
import numpy as np

def predict_disease(img_path):
    # Load and preprocess image
    img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0

    # Make prediction
    predictions = model.predict(img_array)[0]
    
    # Get top 3 predictions
    top_3_indices = predictions.argsort()[-3:][::-1]
    
    print(f"\n📷 Image: {img_path}")
    print("="*50)
    for i in top_3_indices:
        print(f"{class_names[i]} — {predictions[i]*100:.2f}%")
    
    # Confidence threshold check
    top_confidence = predictions[top_3_indices[0]] * 100
    if top_confidence < 75:
        print("\n⚠️ Low confidence! Image may be unclear or not a recognized leaf.")
    else:
        print(f"\n✅ Prediction: {class_names[top_3_indices[0]]} ({top_confidence:.2f}% confident)")

print("Prediction function ready!")

Prediction function ready!


In [7]:
import os

# Pick a sample image from validation set to test
test_folder = "dataset_final/valid/Tomato___Late_blight"
sample_image = os.path.join(test_folder, os.listdir(test_folder)[0])

print("Testing with image:", sample_image)
predict_disease(sample_image)

Testing with image: dataset_final/valid/Tomato___Late_blight\005a2c1f-4e15-49e4-9e5c-61dc3ecf9708___RS_Late.B 5096.JPG
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step

📷 Image: dataset_final/valid/Tomato___Late_blight\005a2c1f-4e15-49e4-9e5c-61dc3ecf9708___RS_Late.B 5096.JPG
Tomato___Late_blight — 99.72%
Tomato___Early_blight — 0.17%
Tomato___Leaf_Mold — 0.09%

✅ Prediction: Tomato___Late_blight (99.72% confident)


In [8]:
# Test with different plant classes
test_images = [
    "dataset_final/valid/Apple___Apple_scab",
    "dataset_final/valid/Potato___healthy",
    "dataset_final/valid/Corn_(maize)___Common_rust_"
]

for folder in test_images:
    sample = os.path.join(folder, os.listdir(folder)[0])
    predict_disease(sample)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step

📷 Image: dataset_final/valid/Apple___Apple_scab\00075aa8-d81a-4184-8541-b692b78d398a___FREC_Scab 3335_270deg.JPG
Apple___Apple_scab — 100.00%
Apple___Cedar_apple_rust — 0.00%
Apple___Black_rot — 0.00%

✅ Prediction: Apple___Apple_scab (100.00% confident)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 184ms/step

📷 Image: dataset_final/valid/Potato___healthy\00fc2ee5-729f-4757-8aeb-65c3355874f2___RS_HL 1864_180deg.JPG
Potato___healthy — 99.93%
Potato___Late_blight — 0.07%
Tomato___healthy — 0.00%

✅ Prediction: Potato___healthy (99.93% confident)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step

📷 Image: dataset_final/valid/Corn_(maize)___Common_rust_\RS_Rust 1564_flipLR.JPG
Corn_(maize)___Common_rust_ — 100.00%
Corn_(maize)___Northern_Leaf_Blight — 0.00%
Tomato___Late_blight — 0.00%

✅ Prediction: Corn_(maize)___Common_rust_ (100.00% confident)


In [6]:
import pickle

# Save history if you still have it
try:
    with open('model/history.pkl', 'wb') as f:
        pickle.dump(history.history, f)
    print("History saved!")
except NameError:
    print("History not available - kernel was restarted")

# Generate predictions on validation set for confusion matrix
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

valid_generator.reset()
y_pred = model.predict(valid_generator, verbose=1)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = valid_generator.classes

print(classification_report(y_true, y_pred_classes, target_names=class_names))

History not available - kernel was restarted
209/209 ━━━━━━━━━━━━━━━━━━━━ 59s 275ms/step
                                     precision    recall  f1-score   support

                 Apple___Apple_scab       0.06      0.07      0.06       504
                  Apple___Black_rot       0.08      0.08      0.08       497
           Apple___Cedar_apple_rust       0.08      0.08      0.08       440
                    Apple___healthy       0.08      0.08      0.08       502
        Corn_(maize)___Common_rust_       0.05      0.05      0.05       477
Corn_(maize)___Northern_Leaf_Blight       0.09      0.09      0.09       477
             Corn_(maize)___healthy       0.08      0.08      0.08       465
              Potato___Early_blight       0.07      0.07      0.07       485
               Potato___Late_blight       0.07      0.06      0.06       485
                   Potato___healthy       0.07      0.07      0.07       456
              Tomato___Early_blight       0.05      0.05      0

In [5]:
from tensorflow.keras.models import load_model
model = load_model('model/plant_model.h5')
class_names = list(train_generator.class_indices.keys())
print("Model loaded! Classes:", class_names)

Model loaded! Classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___healthy']


In [7]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Create a NON-shuffled generator specifically for evaluation
eval_datagen = ImageDataGenerator(rescale=1./255)

eval_generator = eval_datagen.flow_from_directory(
    VALID_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False   # CRITICAL: keeps order matched
)

from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

y_pred = model.predict(eval_generator, verbose=1)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = eval_generator.classes

print(classification_report(y_true, y_pred_classes, target_names=class_names))

Found 6682 images belonging to 14 classes.
209/209 ━━━━━━━━━━━━━━━━━━━━ 58s 277ms/step
                                     precision    recall  f1-score   support

                 Apple___Apple_scab       0.95      0.99      0.97       504
                  Apple___Black_rot       0.98      0.99      0.99       497
           Apple___Cedar_apple_rust       1.00      0.98      0.99       440
                    Apple___healthy       0.99      0.99      0.99       502
        Corn_(maize)___Common_rust_       0.99      1.00      0.99       477
Corn_(maize)___Northern_Leaf_Blight       1.00      0.99      0.99       477
             Corn_(maize)___healthy       1.00      1.00      1.00       465
              Potato___Early_blight       1.00      0.98      0.99       485
               Potato___Late_blight       0.96      0.94      0.95       485
                   Potato___healthy       0.99      0.95      0.97       456
              Tomato___Early_blight       0.93      0.90      0.9

In [8]:
cm = confusion_matrix(y_true, y_pred_classes)
print(cm)

[[497   2   1   1   0   0   0   0   1   0   0   2   0   0]
 [  5 491   1   0   0   0   0   0   0   0   0   0   0   0]
 [  8   0 430   0   1   0   0   0   0   0   0   0   1   0]
 [  5   1   0 495   0   0   0   0   0   0   0   0   0   1]
 [  0   0   0   0 475   2   0   0   0   0   0   0   0   0]
 [  0   0   0   0   2 474   1   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0 465   0   0   0   0   0   0   0]
 [  1   0   0   0   0   0   0 473   6   0   5   0   0   0]
 [  1   1   0   0   0   0   0   1 454   5   1  21   1   0]
 [  3   4   0   2   0   0   0   0  12 434   1   0   0   0]
 [  0   0   0   0   0   0   0   0   0   0 433  26  19   2]
 [  0   0   0   1   0   0   0   0   1   0  11 444   5   1]
 [  0   0   0   0   0   0   0   0   0   0  12   7 451   0]
 [  2   0   0   0   0   0   0   0   0   0   1   2   4 472]]
